# SmartBite YOLO26s-OBB Expiry-Date Detection Fine-Tuning

Fine-tunes Ultralytics `yolo26s-obb.pt` on the same balanced Products date-detection dataset used by the PP-OCRv5 detector notebook.

Workflow:
1. Mount Drive.
2. Unzip `products_date_detection_balanced.zip`.
3. Convert PaddleOCR detection labels into Ultralytics OBB labels.
4. Train `yolo26s-obb.pt`.
5. Validate on val/test splits.
6. Copy the full run and best checkpoint back to Drive.

The dataset remains single-class: `expiry_date`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
!rm -rf /content/dataset /content/yolo_obb_dataset /content/output
!mkdir -p /content/dataset /content/yolo_obb_dataset /content/output


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from PIL import Image


def run_live(cmd, env=None, cwd=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


## Config
Update paths or batch size here if Colab memory says no.


In [ ]:
DATASET_ZIP = Path('/content/drive/My Drive/sb-colab/products_date_detection_balanced.zip')
BASE_UNZIP_DIR = Path('/content/dataset')
YOLO_DATASET_ROOT = Path('/content/yolo_obb_dataset/products_date_detection_obb')

# If this Drive file exists, the notebook uses it. Otherwise Ultralytics will auto-download yolo26s-obb.pt.
YOLO26S_OBB_PT_DRIVE = Path('/content/drive/My Drive/sb-colab/models/yolo26s-obb.pt')
YOLO_MODEL_SOURCE = str(YOLO26S_OBB_PT_DRIVE) if YOLO26S_OBB_PT_DRIVE.exists() else 'yolo26s-obb.pt'

RUNS_PROJECT = Path('/content/output')
RUN_NAME = 'smartbite_yolo26s_obb_expdate_det'
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_expdate_det')
FINAL_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_expdate_det.zip')

EPOCHS = 100
IMGSZ = 1024
BATCH = 4
DEVICE = '0'  # set 'cpu' if no GPU
WORKERS = 2
PATIENCE = 25

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
print('DATASET_ZIP =', DATASET_ZIP)
print('YOLO_MODEL_SOURCE =', YOLO_MODEL_SOURCE)
print('FINAL_MODEL_DRIVE_DIR =', FINAL_MODEL_DRIVE_DIR)


## Unzip PaddleOCR Detection Dataset
This is the same `products_date_detection_balanced.zip` used by `smartbite_ppocrv5_products_date_det_balanced_colab.ipynb`.


In [ ]:
run_live(['unzip', '-q', '-o', DATASET_ZIP, '-d', BASE_UNZIP_DIR])


def find_paddle_det_dataset_root(base: Path) -> Path:
    candidates = [p.parent for p in base.rglob('train_det_label.txt') if (p.parent / 'val_det_label.txt').exists()]
    assert candidates, 'Could not find dataset root containing train_det_label.txt and val_det_label.txt'
    return sorted(candidates, key=lambda p: len(str(p)))[0]


PADDLE_DATASET_ROOT = find_paddle_det_dataset_root(BASE_UNZIP_DIR)
train_label_file = PADDLE_DATASET_ROOT / 'train_det_label.txt'
val_label_file = PADDLE_DATASET_ROOT / 'val_det_label.txt'
test_label_file = PADDLE_DATASET_ROOT / 'test_det_label.txt'
summary_file = PADDLE_DATASET_ROOT / 'summary.json'

for p in [train_label_file, val_label_file, test_label_file, summary_file]:
    print(p, p.exists())
    assert p.exists(), p

if summary_file.exists():
    print(json.dumps(json.loads(summary_file.read_text()), indent=2)[:4000])


## Convert PaddleOCR Labels To Ultralytics OBB
Ultralytics OBB labels use one row per object:

```text
class_index x1 y1 x2 y2 x3 y3 x4 y4
```

The eight coordinates are normalized to `[0, 1]`. The PaddleOCR dataset already stores four polygon points, so this conversion is direct.


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SPLIT_LABEL_FILES = {
    'train': train_label_file,
    'val': val_label_file,
    'test': test_label_file,
}


def clamp01(value: float) -> float:
    return min(1.0, max(0.0, value))


def parse_paddle_label_line(line: str, label_file: Path) -> tuple[Path, list[dict]]:
    rel_path, payload = line.rstrip('
').split('	', 1)
    image_path = label_file.parent / rel_path
    records = json.loads(payload)
    assert isinstance(records, list), f'Label payload must be a list: {label_file}'
    return image_path, records


def normalized_obb_line(points, image_size: tuple[int, int]) -> str | None:
    width, height = image_size
    if width <= 0 or height <= 0:
        return None
    if not isinstance(points, list) or len(points) != 4:
        return None

    coords: list[float] = []
    xs: list[float] = []
    ys: list[float] = []
    for pt in points:
        if not isinstance(pt, (list, tuple)) or len(pt) != 2:
            return None
        x = float(pt[0])
        y = float(pt[1])
        xs.append(x)
        ys.append(y)
        coords.extend([clamp01(x / width), clamp01(y / height)])

    if max(xs) - min(xs) < 2 or max(ys) - min(ys) < 2:
        return None
    return '0 ' + ' '.join(f'{value:.6f}' for value in coords)


def convert_split(split: str, label_file: Path) -> dict[str, int]:
    image_out_dir = YOLO_DATASET_ROOT / 'images' / split
    label_out_dir = YOLO_DATASET_ROOT / 'labels' / split
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    counts = {
        'images': 0,
        'objects': 0,
        'skipped_empty': 0,
        'skipped_missing_image': 0,
        'skipped_bad_polygon': 0,
    }

    for idx, raw_line in enumerate(label_file.read_text(encoding='utf-8').splitlines(), start=1):
        if not raw_line.strip():
            continue
        image_path, records = parse_paddle_label_line(raw_line, label_file)
        if not image_path.exists():
            counts['skipped_missing_image'] += 1
            print(f'[WARN] Missing image at {label_file}:{idx}: {image_path}')
            continue

        with Image.open(image_path) as im:
            width, height = im.size
            ext = image_path.suffix.lower()
            if ext not in IMAGE_EXTS:
                ext = '.jpg'
            out_image_name = f'{image_path.stem}{ext}'
            out_image_path = image_out_dir / out_image_name
            im.convert('RGB').save(out_image_path, quality=95)

        obb_lines: list[str] = []
        for record in records:
            if not isinstance(record, dict):
                counts['skipped_bad_polygon'] += 1
                continue
            line = normalized_obb_line(record.get('points'), (width, height))
            if line is None:
                counts['skipped_bad_polygon'] += 1
                continue
            obb_lines.append(line)

        if not obb_lines:
            counts['skipped_empty'] += 1
            out_image_path.unlink(missing_ok=True)
            continue

        (label_out_dir / f'{Path(out_image_name).stem}.txt').write_text('
'.join(obb_lines) + '
', encoding='utf-8')
        counts['images'] += 1
        counts['objects'] += len(obb_lines)

    return counts


if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)

conversion_summary = {split: convert_split(split, label_file) for split, label_file in SPLIT_LABEL_FILES.items()}
print(json.dumps(conversion_summary, indent=2))


In [ ]:
DATASET_YAML = YOLO_DATASET_ROOT / 'dataset.yaml'
DATASET_YAML.write_text(
    f'''# SmartBite expiry-date OBB dataset converted from PaddleOCR labels
path: {YOLO_DATASET_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: expiry_date
''',
    encoding='utf-8',
)
print(DATASET_YAML.read_text())

for split in ['train', 'val', 'test']:
    images = sorted((YOLO_DATASET_ROOT / 'images' / split).glob('*'))
    labels = sorted((YOLO_DATASET_ROOT / 'labels' / split).glob('*.txt'))
    print(split, 'images:', len(images), 'labels:', len(labels))
    assert images, f'No images for split: {split}'
    assert len(images) == len(labels), f'Image/label mismatch for {split}'


## Install Ultralytics


In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pyyaml'])
run_live(['nvidia-smi'])


## Train YOLO26s-OBB
`yolo26s-obb.pt` is used intentionally here. If the file is not already in Drive, Ultralytics will download it on first use.


In [ ]:
train_script = f'''
from ultralytics import YOLO

model = YOLO(r"{YOLO_MODEL_SOURCE}")
results = model.train(
    data=r"{DATASET_YAML}",
    epochs={EPOCHS},
    imgsz={IMGSZ},
    batch={BATCH},
    device=r"{DEVICE}",
    project=r"{RUNS_PROJECT}",
    name=r"{RUN_NAME}",
    workers={WORKERS},
    patience={PATIENCE},
    cache=False,
    plots=True,
    close_mosaic=10,
)
print(results)
print('Training finished.')
'''

run_live([sys.executable, '-c', train_script])


## Validate Best Checkpoint
Runs validation on both `val` and `test`. The `test` split is useful because the PP-OCRv5 dataset builder exported the real evaluation images there.


In [ ]:
best_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'best.pt'
last_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'last.pt'
assert best_pt.exists(), f'Missing best checkpoint: {best_pt}'
print('best_pt =', best_pt)
print('last_pt =', last_pt, last_pt.exists())

val_script = f'''
from ultralytics import YOLO
model = YOLO(r"{best_pt}")
print('--- VAL ---')
metrics_val = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='val')
print(metrics_val)
print('--- TEST ---')
metrics_test = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='test')
print(metrics_test)
'''
run_live([sys.executable, '-c', val_script])


## Quick Prediction Preview
Saves annotated predictions for a few test images under the run directory so you can sanity-check boxes before downloading.


In [ ]:
preview_script = f'''
from pathlib import Path
from ultralytics import YOLO

model = YOLO(r"{best_pt}")
source = Path(r"{YOLO_DATASET_ROOT}") / 'images' / 'test'
results = model.predict(
    source=str(source),
    imgsz={IMGSZ},
    conf=0.15,
    device=r"{DEVICE}",
    project=r"{RUNS_PROJECT}",
    name=r"{RUN_NAME}_preview",
    save=True,
    max_det=20,
)
print('Preview images saved to:', Path(r"{RUNS_PROJECT}") / '{RUN_NAME}_preview')
print('Predicted images:', len(results))
'''
run_live([sys.executable, '-c', preview_script])


## Backup Artifacts To Drive
Copies the full Ultralytics run directory and creates a zip. The important file is:

```text
/content/drive/My Drive/sb-colab/yolo26s_obb_expdate_det/weights/best.pt
```


In [ ]:
src = RUNS_PROJECT / RUN_NAME
assert src.exists(), f'Missing run dir: {src}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, FINAL_MODEL_DRIVE_DIR)
print('Saved YOLO26s-OBB artifacts to:', FINAL_MODEL_DRIVE_DIR)

if FINAL_ZIP_DRIVE.exists():
    FINAL_ZIP_DRIVE.unlink()
with zipfile.ZipFile(FINAL_ZIP_DRIVE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FINAL_MODEL_DRIVE_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(FINAL_MODEL_DRIVE_DIR.parent))
print('Saved zip:', FINAL_ZIP_DRIVE)
print('Best checkpoint:', FINAL_MODEL_DRIVE_DIR / 'weights' / 'best.pt')


## Notes
- This notebook uses `products_date_detection_balanced.zip`, the same dataset used for PP-OCRv5 detection.
- Labels are converted from PaddleOCR quadrilateral points to Ultralytics OBB `class x1 y1 x2 y2 x3 y3 x4 y4` format.
- The model is single-class: `expiry_date`.
- If Colab runs out of memory, lower `BATCH` first, then lower `IMGSZ` to `768`.
- For local SmartBite integration, the first artifact to try is `weights/best.pt`.
